# Chapter 3 (hadronic) — Notebook 1: Hadronic W reconstruction

**Goals**

- From the light (non-b-tagged) jets, pick the pair whose invariant mass is closest to $m_W$.

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

In [ ]:
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)
events = events[selection.semilep_preselection(events, cuts)]

jets = kinematics.jet_vectors(events)
is_light = events.jet_btag_quantile < cuts.btag_quantile_min
light = jets[is_light]
mask = ak.num(light) >= 2
light = light[mask]

j1, j2, m_jj = pairing.best_W_pair(light)
plt.hist(ak.to_numpy(m_jj), bins=80, range=(0, 200))
plt.axvline(M_W, color='red', label=f'PDG $m_W$ = {M_W:.1f} GeV')
plt.xlabel(r'$m_{jj}$ (best W pair) [GeV]'); plt.legend()

## ✏️ Your turn 1.1

▶️ Change the fit range and re-run.

This fits a Gaussian to the reconstructed hadronic-W peak (`m_jj`) and prints its width σ. Compare σ
to the natural W width (~2 GeV): is the measured width dominated by the W's true width or by the
detector's jet-energy resolution?

> **Challenge (optional):** narrow `FIT_RANGE` around the peak (e.g. `(60, 100)`) and see how σ changes.

In [ ]:
FIT_RANGE = (60, 100)    # ✏️ try (60, 100) to fit just the peak

counts, edges = np.histogram(ak.to_numpy(m_jj), bins=40, range=FIT_RANGE)
result = fitting.fit_gaussian(counts.astype(float), edges)
print(f'reco-W:  mean = {result.params["mu"]:.1f} GeV,  width σ = {result.params["sigma"]:.1f} GeV')

centres = 0.5 * (edges[:-1] + edges[1:])
plt.errorbar(centres, counts, yerr=np.sqrt(np.maximum(counts, 1)), fmt='o', markersize=3, label='reco')
xfit = np.linspace(edges[0], edges[-1], 300)
plt.plot(xfit, fitting.gaussian(xfit, result.params['n'], result.params['mu'], result.params['sigma']), label='fit')
plt.xlabel(r'$m_{jj}$ [GeV]'); plt.ylabel('Events'); plt.legend()